In [0]:
%sql
-- ================================================================
-- SEED: table_config
-- ================================================================
TRUNCATE TABLE bfsi_lakehouse.metadata.table_config;
INSERT INTO bfsi_lakehouse.metadata.table_config (
    source_table_name, source_system, source_format, source_path_pattern,
    target_catalog, target_schema, target_table,
    partition_cols, replace_where_template, drift_policy,
    load_priority, is_active,
    created_at, created_by, last_edited_at, last_edited_by
)
VALUES
('t_Client', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_Client/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_client',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 10, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_AccountCustomer', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_AccountCustomer/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_accountcustomer',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 20, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_Loan', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_Loan/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_loan',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 30, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_LoanInstallment', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_LoanInstallment/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_loaninstallment',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 40, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_AccountTrx', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_AccountTrx/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_accounttrx',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 50, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com');

In [0]:
%sql
-- ================================================================
-- SEED: table_process_config
-- ================================================================
TRUNCATE TABLE bfsi_lakehouse.metadata.table_process_config;
INSERT INTO bfsi_lakehouse.metadata.table_process_config (
    table_id, process_type, load_type,
    depends_on_table_ids, transform_module,
    is_active,
    created_at, created_by, last_edited_at, last_edited_by
)
SELECT
    tc.table_id,
    'BRONZE'              AS process_type,
    'INCREMENTAL'         AS load_type,
    CAST(NULL AS STRING)  AS depends_on_table_ids,  -- Bronze has no intra-layer deps
    CAST(NULL AS STRING)  AS transform_module,      -- Bronze ingests as-is
    TRUE                  AS is_active,
    current_timestamp(), 'rohan.m.mukherjee@gmail.com',
    current_timestamp(), 'rohan.m.mukherjee@gmail.com'
FROM bfsi_lakehouse.metadata.table_config tc
WHERE tc.source_table_name IN (
    't_Client', 't_AccountCustomer', 't_Loan', 't_LoanInstallment', 't_AccountTrx'
);

In [0]:
%sql
-- ================================================================
-- SEED: input_column_config
-- ================================================================
INSERT INTO bfsi_lakehouse.metadata.input_column_config
    (table_id, process_type, target_column_name, source_column_name,
     data_type, is_nullable, is_pii, column_purpose, is_active,
     created_at, created_by, last_edited_at, last_edited_by)

-- ----------------------------------------------------------------
-- t_Client (Silver)
-- ----------------------------------------------------------------
SELECT tc.table_id, 'SILVER',
    col.column_name, col.source_column_name,
    col.data_type, col.is_nullable, col.is_pii,
    col.column_purpose, TRUE,
    current_timestamp(), 'ROHAN', current_timestamp(), 'ROHAN'
FROM bfsi_lakehouse.metadata.table_config tc
CROSS JOIN (VALUES
    ('OurBranchID',  'OurBranchID',  'STRING',    FALSE, FALSE, 'DIMENSION'),
    ('ClientID',     'ClientID',     'STRING',    FALSE, FALSE, 'NATURAL_KEY'),
    ('ClientName',   'ClientName',   'STRING',    FALSE, TRUE,  'ATTRIBUTE'),
    ('ClientType',   'ClientType',   'STRING',    FALSE, FALSE, 'ATTRIBUTE'),
    ('DOB',          'DOB',          'DATE',      TRUE,  TRUE,  'ATTRIBUTE'),
    ('IsActive',     'IsActive',     'BOOLEAN',   FALSE, FALSE, 'FLAG'),
    ('State',        'State',        'STRING',    TRUE,  FALSE, 'ATTRIBUTE'),
    ('Country',      'Country',      'STRING',    TRUE,  FALSE, 'ATTRIBUTE'),
    ('CreatedAt',    'CreatedAt',    'TIMESTAMP', FALSE, FALSE, 'AUDIT'),
    ('CreatedBy',    'CreatedBy',    'STRING',    FALSE, FALSE, 'AUDIT'),
    ('LastUpdatedAt','LastUpdatedAt','TIMESTAMP', TRUE,  FALSE, 'AUDIT'),
    ('dt',           'dt',           'DATE',      FALSE, FALSE, 'PARTITION'),
    ('batch',        'batch',        'INT',       FALSE, FALSE, 'PARTITION')
) AS col(column_name, source_column_name, data_type, is_nullable, is_pii, column_purpose)
WHERE tc.source_table_name = 't_Client'

UNION ALL

-- ----------------------------------------------------------------
-- t_AccountCustomer (Silver)
-- ----------------------------------------------------------------
SELECT tc.table_id, 'SILVER',
    col.column_name, col.source_column_name,
    col.data_type, col.is_nullable, col.is_pii,
    col.column_purpose, TRUE,
    current_timestamp(), 'ROHAN', current_timestamp(), 'ROHAN'
FROM bfsi_lakehouse.metadata.table_config tc
CROSS JOIN (VALUES
    ('OurBranchID',   'OurBranchID',   'STRING',        FALSE, FALSE, 'DIMENSION'),
    ('ClientID',      'ClientID',      'STRING',        FALSE, FALSE, 'FOREIGN_KEY'),
    ('AccountID',     'AccountID',     'STRING',        FALSE, FALSE, 'NATURAL_KEY'),
    ('BranchCode',    'AccountID',     'STRING',        FALSE, FALSE, 'DERIVED'),
    ('ProductCode',   'AccountID',     'STRING',        FALSE, FALSE, 'DERIVED'),
    ('AccountSeq',    'AccountID',     'STRING',        FALSE, FALSE, 'DERIVED'),
    ('ProductID',     'ProductID',     'STRING',        TRUE,  FALSE, 'ATTRIBUTE'),
    ('AccountType',   'AccountType',   'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('ClearBalance',  'ClearBalance',  'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('FreezeAmount',  'FreezeAmount',  'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('AccountStatus', 'AccountStatus', 'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('CreatedAt',     'CreatedAt',     'TIMESTAMP',     FALSE, FALSE, 'AUDIT'),
    ('CreatedBy',     'CreatedBy',     'STRING',        FALSE, FALSE, 'AUDIT'),
    ('LastUpdatedAt', 'LastUpdatedAt', 'TIMESTAMP',     TRUE,  FALSE, 'AUDIT'),
    ('dt',            'dt',            'DATE',          FALSE, FALSE, 'PARTITION'),
    ('batch',         'batch',         'INT',           FALSE, FALSE, 'PARTITION')
) AS col(column_name, source_column_name, data_type, is_nullable, is_pii, column_purpose)
WHERE tc.source_table_name = 't_AccountCustomer'

UNION ALL

-- ----------------------------------------------------------------
-- t_Loan (Silver)
-- ----------------------------------------------------------------
SELECT tc.table_id, 'SILVER',
    col.column_name, col.source_column_name,
    col.data_type, col.is_nullable, col.is_pii,
    col.column_purpose, TRUE,
    current_timestamp(), 'ROHAN', current_timestamp(), 'ROHAN'
FROM bfsi_lakehouse.metadata.table_config tc
CROSS JOIN (VALUES
    ('OurBranchID',          'OurBranchID',         'STRING',        FALSE, FALSE, 'DIMENSION'),
    ('AccountID',            'AccountID',            'STRING',        FALSE, FALSE, 'FOREIGN_KEY'),
    ('LoanID',               'LoanID',               'STRING',        FALSE, FALSE, 'NATURAL_KEY'),
    ('LoanSeries',           'LoanSeries',           'INT',           FALSE, FALSE, 'ATTRIBUTE'),
    ('SanctionDate',         'SanctionDate',         'DATE',          TRUE,  FALSE, 'ATTRIBUTE'),
    ('SanctionAmount',       'SanctionAmount',       'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('DisbursementDate',     'DisbursementDate',     'DATE',          TRUE,  FALSE, 'ATTRIBUTE'),
    ('DisbursementAmount',   'DisbursementAmount',   'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('LoanStatus',           'LoanStatus',           'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('InterestRate',         'InterestRate',         'DECIMAL(6,4)',  TRUE,  FALSE, 'DERIVED'),
    ('InterestAmount',       'InterestAmount',       'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('Tenure',               'Tenure',               'INT',           TRUE,  FALSE, 'ATTRIBUTE'),
    ('Frequency',            'Frequency',            'STRING',        TRUE,  FALSE, 'ATTRIBUTE'),
    ('RepaymentType',        'RepaymentType',        'STRING',        TRUE,  FALSE, 'ATTRIBUTE'),
    ('OutstandingPrincipal', 'OutstandingPrincipal', 'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('OutstandingInterest',  'OutstandingInterest',  'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('DPD',                  'DPD',                  'INT',           TRUE,  FALSE, 'DERIVED'),
    ('IsNPS',                'IsNPS',                'BOOLEAN',       TRUE,  FALSE, 'FLAG'),
    ('ParFlag',              'ParFlag',              'BOOLEAN',       TRUE,  FALSE, 'FLAG'),
    ('FundID',               'FundID',               'STRING',        TRUE,  FALSE, 'DIMENSION'),
    ('MaturityDate',         'MaturityDate',         'DATE',          TRUE,  FALSE, 'ATTRIBUTE'),
    ('CreatedAt',            'CreatedAt',            'TIMESTAMP',     FALSE, FALSE, 'AUDIT'),
    ('CreatedBy',            'CreatedBy',            'STRING',        FALSE, FALSE, 'AUDIT'),
    ('LastUpdatedAt',        'LastUpdatedAt',        'TIMESTAMP',     TRUE,  FALSE, 'AUDIT'),
    ('dt',                   'dt',                   'DATE',          FALSE, FALSE, 'PARTITION'),
    ('batch',                'batch',                'INT',           FALSE, FALSE, 'PARTITION')
) AS col(column_name, source_column_name, data_type, is_nullable, is_pii, column_purpose)
WHERE tc.source_table_name = 't_Loan'

UNION ALL

-- ----------------------------------------------------------------
-- t_LoanInstallment (Silver)
-- ----------------------------------------------------------------
SELECT tc.table_id, 'SILVER',
    col.column_name, col.source_column_name,
    col.data_type, col.is_nullable, col.is_pii,
    col.column_purpose, TRUE,
    current_timestamp(), 'ROHAN', current_timestamp(), 'ROHAN'
FROM bfsi_lakehouse.metadata.table_config tc
CROSS JOIN (VALUES
    ('OurBranchID',       'OurBranchID',       'STRING',        FALSE, FALSE, 'DIMENSION'),
    ('AccountID',         'AccountID',         'STRING',        FALSE, FALSE, 'FOREIGN_KEY'),
    ('LoanID',            'LoanID',            'STRING',        FALSE, FALSE, 'FOREIGN_KEY'),
    ('LoanSeries',        'LoanSeries',        'INT',           FALSE, FALSE, 'ATTRIBUTE'),
    ('InstallmentNo',     'InstallmentNo',     'INT',           FALSE, FALSE, 'NATURAL_KEY'),
    ('InstallmentDate',   'InstallmentDate',   'DATE',          FALSE, FALSE, 'ATTRIBUTE'),
    ('PrincipalDue',      'PrincipalDue',      'DECIMAL(15,2)', FALSE, FALSE, 'DERIVED'),
    ('InterestDue',       'InterestDue',       'DECIMAL(15,2)', FALSE, FALSE, 'DERIVED'),
    ('InstallmentAmount', 'InstallmentAmount', 'DECIMAL(15,2)', FALSE, FALSE, 'DERIVED'),
    ('PaidStatus',        'PaidStatus',        'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('TransactionDate',   'TransactionDate',   'DATE',          TRUE,  FALSE, 'ATTRIBUTE'),
    ('CreatedAt',         'CreatedAt',         'TIMESTAMP',     FALSE, FALSE, 'AUDIT'),
    ('CreatedBy',         'CreatedBy',         'STRING',        FALSE, FALSE, 'AUDIT'),
    ('LastUpdatedAt',     'LastUpdatedAt',     'TIMESTAMP',     TRUE,  FALSE, 'AUDIT'),
    ('dt',                'dt',                'DATE',          FALSE, FALSE, 'PARTITION'),
    ('batch',             'batch',             'INT',           FALSE, FALSE, 'PARTITION')
) AS col(column_name, source_column_name, data_type, is_nullable, is_pii, column_purpose)
WHERE tc.source_table_name = 't_LoanInstallment'

UNION ALL

-- ----------------------------------------------------------------
-- t_AccountTrx (Silver)
-- ----------------------------------------------------------------
SELECT tc.table_id, 'SILVER',
    col.column_name, col.source_column_name,
    col.data_type, col.is_nullable, col.is_pii,
    col.column_purpose, TRUE,
    current_timestamp(), 'ROHAN', current_timestamp(), 'ROHAN'
FROM bfsi_lakehouse.metadata.table_config tc
CROSS JOIN (VALUES
    ('OurBranchID',        'OurBranchID',        'STRING',        FALSE, FALSE, 'DIMENSION'),
    ('AccountID',          'AccountID',          'STRING',        FALSE, FALSE, 'FOREIGN_KEY'),
    ('LoanID',             'LoanID',             'STRING',        TRUE,  FALSE, 'FOREIGN_KEY'),
    ('LoanSeries',         'LoanSeries',         'INT',           TRUE,  FALSE, 'ATTRIBUTE'),
    ('TrxID',              'TrxID',              'STRING',        FALSE, FALSE, 'NATURAL_KEY'),
    ('TrxDateTime',        'TrxDateTime',        'TIMESTAMP',     FALSE, FALSE, 'ATTRIBUTE'),
    ('ValueDate',          'ValueDate',          'DATE',          FALSE, FALSE, 'ATTRIBUTE'),
    ('TrxType',            'TrxType',            'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('DrCr',               'DrCr',               'STRING',        FALSE, FALSE, 'ATTRIBUTE'),
    ('Amount',             'Amount',             'DECIMAL(15,2)', FALSE, FALSE, 'DERIVED'),
    ('PrincipalComponent', 'PrincipalComponent', 'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('InterestComponent',  'InterestComponent',  'DECIMAL(15,2)', TRUE,  FALSE, 'DERIVED'),
    ('PaymentMode',        'PaymentMode',        'STRING',        TRUE,  FALSE, 'ATTRIBUTE'),
    ('CreatedAt',          'CreatedAt',          'TIMESTAMP',     FALSE, FALSE, 'AUDIT'),
    ('CreatedBy',          'CreatedBy',          'STRING',        FALSE, FALSE, 'AUDIT'),
    ('LastUpdatedAt',      'LastUpdatedAt',      'TIMESTAMP',     TRUE,  FALSE, 'AUDIT'),
    ('dt',                 'dt',                 'DATE',          FALSE, FALSE, 'PARTITION'),
    ('batch',              'batch',              'INT',           FALSE, FALSE, 'PARTITION')
) AS col(column_name, source_column_name, data_type, is_nullable, is_pii, column_purpose)
WHERE tc.source_table_name = 't_AccountTrx';

In [0]:
%sql
ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    DROP CONSTRAINT icc_chk_purpose;

ALTER TABLE bfsi_lakehouse.metadata.input_column_config
    ADD CONSTRAINT icc_chk_purpose
    CHECK (column_purpose IN (
        'NATURAL_KEY', 'FOREIGN_KEY', 'ATTRIBUTE',
        'MEASURE', 'FLAG', 'DERIVED', 'AUDIT', 'DIMENSION', 'PARTITION'
    ));